# Merge original and new feature CSVs

This notebook combines the original and new-only feature CSVs separately for train, dev, and test. It validates their schemas, refuses duplicate composer/source-path rows, sorts the result by composer, and writes new combined files without modifying either input set.


In [1]:
from pathlib import Path

import pandas as pd

FEATURE_DIR = Path('../data/features')
SPLITS = ('train', 'dev', 'test')
ORIGINAL_PATHS = {split: FEATURE_DIR / 'original features' / f'{split}_features.csv' for split in SPLITS}
NEW_PATHS = {split: FEATURE_DIR / f'new_{split}_features.csv' for split in SPLITS}
COMBINED_PATHS = {split: FEATURE_DIR / f'combined_{split}_features.csv' for split in SPLITS}


In [2]:
for split in SPLITS:
    if not ORIGINAL_PATHS[split].is_file():
        raise FileNotFoundError(f'Missing original {split} CSV: {ORIGINAL_PATHS[split]}')
    if not NEW_PATHS[split].is_file():
        raise FileNotFoundError(f'Missing new {split} CSV: {NEW_PATHS[split]}')

    original = pd.read_csv(ORIGINAL_PATHS[split])
    new = pd.read_csv(NEW_PATHS[split])
    if original.columns.tolist() != new.columns.tolist():
        raise ValueError(f'Column mismatch in the {split} CSVs; do not merge them until their schemas match.')

    combined = pd.concat([original, new], ignore_index=True)
    duplicate_rows = combined.duplicated(['composer', 'relative_path'], keep=False)
    if duplicate_rows.any():
        duplicates = combined.loc[duplicate_rows, ['composer', 'relative_path', 'filename']]
        raise ValueError(f'Found duplicate composer/source-path rows in {split}:\n{duplicates.to_string(index=False)}')

    combined = combined.sort_values(['composer', 'relative_path', 'filename'], kind='stable', ignore_index=True)
    combined.to_csv(COMBINED_PATHS[split], index=False)
    print(f'{split}: {len(original):,} original + {len(new):,} new = {len(combined):,} rows -> {COMBINED_PATHS[split]}')


train: 1,138 original + 412 new = 1,550 rows -> ..\data\features\combined_train_features.csv
dev: 242 original + 109 new = 351 rows -> ..\data\features\combined_dev_features.csv
test: 248 original + 90 new = 338 rows -> ..\data\features\combined_test_features.csv
